# ⚙️ Configuration & Export

How to configure parsing, mask proper nouns, and build multi-project datasets.

## 🎯 Goals of this notebook

1. **Understand RunConfig options** — learn every toggle the parser exposes
2. **Compare outputs** — see how dropping damaged signs or masking POS changes transliterations
3. **Build a multi-project dataset** — parse several corpora and combine into one DataFrame
4. **Export** — save combined data as JSONL and CSV for downstream use
5. **Quick analysis** — explore the combined dataset by project, provenance, and period

## Configuration Options

Before we start, if the default data paths aren't correct for your machine, you can re-configure them:
By default, `DATA_DIR` is set to `<repo_root>/enriched_data`. If you have it somewhere else (like `D:/thesis_data`):

In [1]:
import oracc_parser.settings as settings
from pathlib import Path

# settings.DATA_DIR = Path("D:/my_custom_oracc_data")

## 1. RunConfig options

| Option | Default | What it does |
|---|---|---|
| `limit` | `None` | Only parse first N texts (good for testing) |
| `max_break_fraction` | `1.0` | **Word-level**: fraction of broken signs tolerated before a word is replaced with `X` in transliteration / normalization / lemmatization |
| `drop_missing` | `False` | **Sign-level**: replace signs marked `[x]` (completely broken) with `X` in **Unicode cuneiform output only** |
| `drop_damaged` | `False` | **Sign-level**: replace signs marked `⸢x⸣` (partially damaged) with `X` in **Unicode cuneiform output only** |
| `mask_pos` | `[]` | Replace words of certain POS with the tag name |
| `languages` | `["Akkadian"]` | Which languages to process |
| `USE_CACHE` | `True` | Use cached results if available |

### ✅ Valid Config Values

When configuring a parser run with `RunConfig` (as seen in Notebook 01), you can filter by `languages` or `mask_pos`. The available values are exactly those listed in the reference tables below:

- **`languages`**: Any language listed in the `Languages` table (e.g., `Akkadian`, `Sumerian`, `Hittite`, `Ugaritic`, or simply `all` to include everything).
- **`mask_pos`**: Any POS tag from the `pos_tags` table below (e.g., `PN`, `DN`, `RN`, `N`, `V`).
- **`provenance` / `period`**: If filtering results programmatically by these fields in your analysis, use the `normalized_city` from the provenances table or the `period_name` from the periods table.

If you want to know what values are valid for `mask_pos`, `languages`, or `periods`, you can query the bundled reference data directly:

In [1]:
from oracc_parser.pipeline import reference_data

# See valid POS tags for masking
pos_df = reference_data.get_pos_tags()
print('Valid POS tags (first 15):')
print(pos_df['POS-tag'].head(15).tolist())
print()

# See valid languages
lang_df = reference_data.get_languages()
print('Valid Languages:')
print(lang_df['lang'].tolist())

Valid POS tags (first 15):
['MN', 'n', 'u', 'N', 'CN', 'AJ', 'PRP', 'V', 'IP', 'PN', 'MOD', 'RN', 'GN', 'REL', 'AV']

Valid Languages:
['sux', 'akk', 'akk-x-neoass', 'akk-x-stdbab', 'akk-x-neobab', 'akk-x-midass', 'akk-x-mbperi', 'qpc', 'akk-x-ltebab', 'akk-x-oldbab', 'akk-949', 'uga-040', 'sux-x-emesal', 'xur', 'peo', 'xur-946', 'hit', 'akk-x-midbab', 'arc', 'arc-949', 'elx', 'akk-x-stdbab-949', 'akk-x-earakk', 'akk-x-oldass', 'uga', 'xhu', 'hit-946', 'akk-936', 'xur-944', 'qca', 'qeb', 'akk-x-mbperi-949', 'akk-x-oldakk', 'qcu', 'qpe', 'grc', 'qur', 'akk-935', 'xhu-040', 'akk-x-neoass-949', 'xhu-946', 'sux-947', 'egy-020', 'sux-x-gloss', 'hlu', 'akk-x-neobab-949', 'sux-x-syll', 'hit-040', 'akk-x-oldbab-949']


> ⚠️ Important note: not all ORACC projects have POS-tags for every word in their corpus. When unknown, this field remains empty.

## 2. Word-level vs sign-level break filtering

`RunConfig` exposes **two independent levels** of break filtering that operate on different granularities and affect different outputs.

### Word-level — `max_break_fraction` (0.0 – 1.0)

- Each word has a `break_perc`: the **fraction of its signs** that are broken or missing (averaged across all signs in the word).

$\text{break\_perc} = \frac{n_{\text{broken}} + 0.5 \cdot n_{\text{damaged}}}{n_{\text{total}}}$

- Words whose `break_perc` **exceeds** `max_break_fraction` are replaced wholesale with `X`.
- Affects → **transliteration**, **normalization**, **lemmatization**.
- `1.0` (default) → keep all words regardless of damage.
- `0.0` → replace any word that has even one broken sign.

### Sign-level — `drop_missing` / `drop_damaged`

- Operates **sign-by-sign**, not word-by-word.
- `drop_missing=True` replaces signs marked `[x]` (completely lost) with `X`.
- `drop_damaged=True` replaces signs marked `⸢x⸣` (partially legible) with `X`.
- Affects → **Unicode cuneiform output only**.

> ⚠️ **The two levels are independent and produce results that may not align.**
> A word kept intact in the transliteration (because its *average* damage is below `max_break_fraction`) may still have individual signs stripped from the Unicode cuneiform output by `drop_missing` / `drop_damaged`, and vice-versa. Do not expect the text outputs and the Unicode cuneiform to be aligned token-for-token when using these parameters.

In [1]:
# max_break_fraction operates on the word level only
# It has no effect on Unicode cuneiform
from oracc_parser import parse_project_from_word_csvs, RunConfig, get_transliterations, get_unicode_texts
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR

PROJECT = "babcity"
project_csv_dir = WORD_CSV_DIR / PROJECT.replace("/", "-")

# Load a 2-text sample
all_word_dfs = load_word_csvs_from_dir(project_csv_dir, project=PROJECT)
sample_dfs = dict(list(all_word_dfs.items())[:2])

# Strict word-level filtering: words with >30% broken signs → replaced with X
rec_strict = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(fetch_translations=False, max_break_fraction=0.3))

# Liberal (default): keep all words regardless of damage
rec_liberal = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(fetch_translations=False, max_break_fraction=1.0))

print("=== Transliteration with max_break_fraction=0.3 (strict) ===")
for _, r in get_transliterations(rec_strict).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Transliteration with max_break_fraction=1.0 (liberal, default) ===")
for _, r in get_transliterations(rec_liberal).iterrows():
    print(f"  {r['id']}:")
    print(f"    {r['transliteration'][:120]}")
    print(f"    tokens_without_broken: {r['tokens_without_broken']}/{r['total_tokens']}")

print()
print("=== Unicode cuneiform is unaffected by max_break_fraction ===")
print("(drop_missing / drop_damaged control sign-level filtering here)")
for _, r in get_unicode_texts(rec_strict).iterrows():
    print(f"  {r['id']}: {r['unicode'][:80]}")

12:04:34 | INFO    | oracc_parser | Loaded 224 word CSV(s) from G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs\babcity
Processing babcity:   0%|          | 0/2 [00:00<?, ?it/s]c:\Users\user\anaconda3\envs\oracc-parser\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oracc.museum.upenn.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Processing babcity:  50%|█████     | 1/2 [00:01<00:01,  1.27s/it]c:\Users\user\anaconda3\envs\oracc-parser\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oracc.museum.upenn.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Processing babcity: 100%|██████████| 2/2 [00:02<00:00,  1.23s/it]
12:04:3

=== Transliteration with max_break_fraction=0.3 (strict) ===
  babcity_P261352:
    X X X X X X X X X
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ X
A ša₂ {m}{d}EN.LIL₂-TIN{iṭ} ša₂ ina IGI {m}ti-ri-ra-ka-am-ma
DUMU E₂ š
    tokens_without_broken: 63/88
  babcity_P285916:
    E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik
ša₂ DA E₂ {m}SILA-a-a
{m}ni-din-tu₄ DUMU {m}ši-rik A {m}e-gi₃-bi
a-na i-di E₂ a
    tokens_without_broken: 91/92

=== Transliteration with max_break_fraction=1.0 (liberal, default) ===
  babcity_P261352:
    1 1/2 MA.NA KU₃.BABBAR i-di E₂ ša₂ MU 1-KAM]
{m}da-ri-ia-⸢a⸣-muš LUGAL ša₂ {[m}{d}EN-it-tan-nu]
A ša₂ {m}{d}EN.LIL₂-TIN{
    tokens_without_broken: 88/88
  babcity_P285916:
    E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik
ša₂ DA E₂ {m}SILA-a-a
{m}ni-din-tu₄ DUMU {m}ši-rik A {m}e-gi₃-bi
a-na i-di E₂ a
    tokens_without_broken: 92/92

=== Unicode cuneiform is unaffected by max_break_fraction ===
(drop_missing / drop_damaged control sign-level filtering here)
  babcity_P261352: 𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 

In [3]:
# drop_missing / drop_damaged operate sign-by-sign on Unicode cuneiform only
# They have no effect on transliteration, normalization, or lemmatization
rec_drop_none    = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(fetch_translations=False, drop_missing=False, drop_damaged=False))
rec_drop_missing = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(fetch_translations=False, drop_missing=True,  drop_damaged=False))
rec_drop_both    = parse_project_from_word_csvs(PROJECT, sample_dfs, config=RunConfig(fetch_translations=False, drop_missing=True,  drop_damaged=True))

from oracc_parser import get_normalizations

for label, recs in [
    ("drop_missing=False, drop_damaged=False (default)", rec_drop_none),
    ("drop_missing=True",                               rec_drop_missing),
    ("drop_missing=True, drop_damaged=True",            rec_drop_both),
]:
    print(f"=== {label} ===")
    r = get_unicode_texts(recs).iloc[0]
    print(f"  Unicode:        {r['unicode'][:120]}")
    r2 = get_normalizations(recs).iloc[0]
    print(f"  Normalization:  {r2['normalization'][:120]}  (unchanged)")
    print()


Processing babcity: 100%|██████████| 2/2 [00:00<00:00,  4.91it/s]
12:10:29 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
Processing babcity: 100%|██████████| 2/2 [00:00<00:00,  5.93it/s]
12:10:29 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs
Processing babcity: 100%|██████████| 2/2 [00:00<00:00,  7.38it/s]
12:10:30 | INFO    | oracc_parser | Processed 2 tablets for babcity from word CSVs


=== drop_missing=False, drop_damaged=False (default) ===
  Unicode:        𒁹 𒈦 𒈠𒈾 𒆬𒌓 𒄿𒁲 𒂍 𒃻 𒈬 𒁹𒄰
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭𒈦𒋀𒌶 𒇽𒀴 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀸 𒋗𒈫 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒈠𒆟 𒂊𒋛𒀀 𒌑𒃻𒊍𒍝
  Normalization:  UNK UNK manû kaspi idī bīti ša šatti UNK
Darius šarri ša Bel-ittannu
māru ša Enlil-uballiṭ ša ina pāni Tirirakamma
mār b  (unchanged)

=== drop_missing=True ===
  Unicode:        X X XX XX XX X X X XX
𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 XXXXXX
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭𒈦𒋀𒌶 𒇽𒀴 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀸 𒋗𒈫 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒈠𒆟 𒂊𒋛𒀀 𒌑𒃻𒊍𒍝
  Normalization:  UNK UNK manû kaspi idī bīti ša šatti UNK
Darius šarri ša Bel-ittannu
māru ša Enlil-uballiṭ ša ina pāni Tirirakamma
mār b  (unchanged)

=== drop_missing=True, drop_damaged=True ===
  Unicode:        X X XX XX XX X X X XX
𒁹𒁕𒊑𒅀X𒈲 𒈗 𒃻 XXXXXX
𒀀 𒃻 𒁹𒀭𒂗𒆤𒁷𒀉 𒃻 𒀸 𒅆 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒌉 𒂍 𒃻 𒁹𒀭𒂗𒆤𒈬𒈬
𒁹𒀭𒈦𒋀𒌶 𒇽𒀴 𒃻 𒁹𒀭𒂗𒀉𒆗𒉡
𒀸 𒋗𒈫 𒁹𒋾𒊑𒊏𒅗𒄠𒈠
𒈠𒆟 𒂊𒋛𒀀 𒌑𒃻𒊍𒍝
  Normalization:  UNK UNK manû kaspi idī bīti ša šatti UNK
Darius šarri ša Bel-ittannu
māru ša Enlil-uballiṭ ša ina pāni Tirirakamma
mār b  (unchange

## 3. Build a multi-project dataset

In [4]:
# Parse a few projects and combine
import pandas as pd
from oracc_parser import parse_project_from_word_csvs, RunConfig, get_full_flat_table, get_full_flat_table, get_metadata_table
from oracc_parser.io.word_csv import load_word_csvs_from_dir
from oracc_parser.settings import WORD_CSV_DIR, OUTPUT_DIR

PROJECTS = ["babcity", "borsippa"]  # change these to what you want
config = RunConfig(fetch_translations=False, max_break_fraction=.5, drop_missing=True)

all_dfs = []
for project in PROJECTS:
    print(f"Parsing {project}...")
    try:
        project_csv_dir = WORD_CSV_DIR / project.replace("/", "-")
        word_dfs = load_word_csvs_from_dir(project_csv_dir, project=project)
        # Limit to first 3 texts for demo; remove slice for full parse
        sample_dfs = dict(list(word_dfs.items())[:3])
        records = parse_project_from_word_csvs(project, sample_dfs, config=config)
        df = get_full_flat_table(records)
        all_dfs.append(df)
        print(f"  -> {len(records)} tablets")
    except Exception as e:
        print(f"  -> Error: {e}")

combined = pd.concat(all_dfs, ignore_index=True)
print(f"Combined: {len(combined)} tablets from {len(PROJECTS)} projects")
display(combined)

Parsing babcity...


12:11:24 | INFO    | oracc_parser | Loaded 224 word CSV(s) from G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs\babcity
Processing babcity:  67%|██████▋   | 2/3 [00:00<00:00,  6.10it/s]c:\Users\user\anaconda3\envs\oracc-parser\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oracc.museum.upenn.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Processing babcity: 100%|██████████| 3/3 [00:01<00:00,  1.83it/s]
12:11:26 | INFO    | oracc_parser | Processed 3 tablets for babcity from word CSVs


  -> 3 tablets
Parsing borsippa...


borsippa.zip: 100%|██████████| 834k/834k [00:00<00:00, 1.99MB/s]


  -> Extracted borsippa projects to G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs


12:11:47 | INFO    | oracc_parser | Loaded 224 word CSV(s) from G:\My Drive\GitHub\oracc-parser\enriched_data\oracc_csvs\borsippa
Processing borsippa:   0%|          | 0/3 [00:00<?, ?it/s]c:\Users\user\anaconda3\envs\oracc-parser\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oracc.museum.upenn.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Processing borsippa:  33%|███▎      | 1/3 [00:01<00:02,  1.25s/it]c:\Users\user\anaconda3\envs\oracc-parser\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'oracc.museum.upenn.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
Processing borsippa:  67%|██████▋   | 2/3 [00:02<00:01,  1.23s/it]c:\U

  -> 3 tablets
Combined: 6 tablets from 2 projects


,id,project,text_id,genre,archive,provenance,period,start_year,end_year,transliteration,normalization,lemmatization,unicode,translation,total_tokens,tokens_without_broken
0,babcity_P261352,babcity,P261352,Legal,Murašû Archive,Nippur,achaemenid,-550,-330,X X X X X X X X X\n{m}da-ri-ia-⸢a⸣-muš LUGAL š...,X X X X X X X X X\nDarius šarri ša X\nmāru ša ...,X X X X X X X X X\nDarius šarru ša X\nmāru ša ...,X X XX XX XX X X X XX\n𒁹𒁕𒊑𒅀𒀀𒈲 𒈗 𒃻 XXXXXX\n𒀀 𒃻 ...,"[1 1/2 minas of silver, the house rent from th...",88,64
1,babcity_P285916,babcity,P285916,Legal,Egibi Archive,Babylon,achaemenid,-550,-330,E₂ ša₂ {m}ni-din-ti-{d}EN DUMU {m}ši-rik\nša₂ ...,bītu ša Nidinti-Bel māri Širku\nša ṭāh bīt Suq...,bītu ša Nidinti-Bel māru Širku\nša ṭāh bītu Su...,𒂍 𒃻 𒁹𒉌𒁷𒋾𒀭𒂗 𒌉 𒁹𒅆𒋆\n𒃻 𒁕 𒂍 𒁹𒋻𒀀𒀀\n𒁹𒉌𒁷𒌈 𒌉 𒁹𒅆𒋆 𒀀 𒁹𒂊𒐕...,"A house of Nidinti-Bel, son of Širku, which is...",92,92
2,babcity_P295135,babcity,P295135,Legal,Ea-ilutu-bani Archive,Borsippa,achaemenid,-550,-330,E₂-⸢su⸣ ša₂ {e₂}a-su-up ša₂ {m}⸢x⸣-[...]\nA-šu...,bīssu ša asup ša X\nmārišu ša Luṣi-ana-nur-Mar...,bītu ša asuppu ša X\nmāru ša Luṣi-ana-nur-Mard...,𒂍𒋢 𒃻 𒂍𒀀𒋢𒌒 𒃻 𒁹XX\n𒀀𒋙 𒃻 𒁹𒇻𒍮𒀀𒈾𒂟𒀭𒀫𒌓 X\n𒀀𒈾 𒈬𒀭𒈾 𒈫 𒂆 ...,(Concerning) the with an out-building belongin...,84,70
3,borsippa_P202111,borsippa,P202111,Legal,Ilia Archive,Borsippa,achaemenid,-550,-330,2 1/2 u₄-mu maš-ti-ti ša₂ {iti}GUD 2 1/2 u₄-mu...,UNK UNK ūmu maštīti ša Ayyaru UNK UNK ūmu\nmaš...,UNK UNK ūmu maštītu ša Ayyaru UNK UNK ūmu\nmaš...,𒈫 𒈦 𒌓𒈬 𒈦𒋾𒋾 𒃻 𒌗𒄞 𒈫 𒈦 𒌓𒈬\nX𒋾𒋾 𒌗𒆥 𒈫 𒈦 𒌓𒈬 𒆏𒊑 𒃻 𒌗𒃶\...,2½ days maštītu in month II\n2½ days maštītu (...,102,91
4,borsippa_P202351,borsippa,P202351,Legal,Ilia Archive,Borsippa,Neo-Babylonian,-626,-539,X X X {m}ṣil-la-a A {m}DINGIR-⸢ia⸣\nX X lib₃]-...,X X X Ṣilla mār Ilia\nX X libbišu Kalba qallaš...,X X X Ṣilla māru Ilia\nX X libbu Kalba qallu\n...,XXXX XX X 𒁹𒉣𒆷𒀀 𒀀 𒁹𒀭𒅀\nX XX X𒁉𒋙 𒁹𒆗𒁀𒀀 𒇽𒃲𒆷X\nXX 𒇽...,[Šulā son of] Ṣillā of the Ilia family has [vo...,157,96
5,borsippa_P202504,borsippa,P202504,Legal,Sīrāšû Archive,Borsippa,achaemenid,-550,-330,X u₄]-mu tar-din-nu {iti}ŠE TA UD [x]-KAM\nX ⸢...,X ūmu tardinnu Addaru ultu ūm UNK\nX ūm UNK UN...,X ūmu tardennu Addaru ištu ūmu UNK\nX ūmu UNK ...,X X𒈬 𒋻𒁷𒉡 𒌗𒊺 𒋫 𒌓 X𒄰\nXX 𒌓 𒌋𒐋𒄰 𒐈 𒌓𒈬 𒋾𒅆𒆕𒈨𒌍\nX𒁈 𒌓 ...,[x da]ys tardennu (offering) from [day x until...,247,181


In [5]:
# Export the combined dataset
out = OUTPUT_DIR
out.mkdir(parents=True, exist_ok=True)

# change the file names here, if you want
path_jsonl = out / "combined_dataset.jsonl"
path_csv = out / "combined_dataset.csv"

combined.to_json(path_jsonl, orient="records", lines=True, force_ascii=False)
combined.to_csv(path_csv, index=False)

print(f"Saved to:")
print(f"  {path_jsonl}  ({path_jsonl.stat().st_size/1024:.1f} KB)")
print(f"  {path_csv}  ({path_csv.stat().st_size/1024:.1f} KB)")
print(f"\n📁 All output files in {out}:")
for f in sorted(out.iterdir()):
    if f.is_file():
        print(f"  {f.name:40s}  {f.stat().st_size/1024:.1f} KB")

Saved to:
  G:\My Drive\GitHub\oracc-parser\enriched_data\output\combined_dataset.jsonl  (30.1 KB)
  G:\My Drive\GitHub\oracc-parser\enriched_data\output\combined_dataset.csv  (28.4 KB)

📁 All output files in G:\My Drive\GitHub\oracc-parser\enriched_data\output:
  babcity.csv                               16.1 KB
  babcity.jsonl                             17.4 KB
  combined_dataset.csv                      28.4 KB
  combined_dataset.jsonl                    30.1 KB


## 5. Quick analysis of the data

In [6]:
print("Texts by project:")
print(combined["project"].value_counts().to_string())

print("\nTexts by provenance:")
print(combined["provenance"].value_counts().to_string())

print("\nTexts by period:")
print(combined["period"].value_counts().to_string())

print(f"\nAvg tokens per text: {combined['total_tokens'].mean():.0f}")
print(f"Total tokens: {combined['total_tokens'].sum()}")

Texts by project:
project
babcity     3
borsippa    3

Texts by provenance:
provenance
Borsippa    4
Nippur      1
Babylon     1

Texts by period:
period
achaemenid        5
Neo-Babylonian    1

Avg tokens per text: 128
Total tokens: 770


## What's next?

- **Quickstart:** See `01_quickstart.ipynb` to quickly download data, parse a project from CSVs, and export results
- **Explore what's in the dataset:** See notebook `02_reference_data.ipynb` for a full overview of collected projects, catalogues, provenance, periods, and other reference data.
- **understand the process:** see notebook `04_oracc_json_processing.ipynb` for in-depth explanations on how the package processes the original oracc files and metadata.